# Laboratório 03 — Completando o ACO

**Objetivo:** implementar as partes que faltam no ACO, usando como referência o código já estudado no Laboratório 01.

As lacunas (`TODO`) foram preenchidas com a mesma lógica vista em aula. Os trechos que já vinham prontos no enunciado (importações, matriz de custos, parâmetros e `obter_vizinhos`) foram mantidos exatamente como no roteiro.


In [1]:
%matplotlib inline

## 1. Representação da rede

In [2]:
import numpy as np
import random
import matplotlib.pyplot as plt

CUSTOS = np.array([
    [0, 2, 4, np.inf, np.inf, np.inf],
    [2, 0, 1, 5, np.inf, np.inf],
    [4, 1, 0, 2, 3, np.inf],
    [np.inf, 5, 2, 0, 1, 4],
    [np.inf, np.inf, 3, 1, 0, 2],
    [np.inf, np.inf, np.inf, 4, 2, 0]
])

ORIGEM = 0
DESTINO = 5

NUM_FORMIGAS = 20
NUM_ITERACOES = 50

ALPHA = 1.0
BETA = 2.0

TAXA_EVAPORACAO = 0.5
Q = 100

feromonio = np.ones_like(CUSTOS, dtype=float)
feromonio[CUSTOS == np.inf] = 0

## 2. Função de vizinhos

Esta parte já vinha pronta no roteiro.

In [3]:
def obter_vizinhos(no):

    vizinhos = []

    for proximo in range(len(CUSTOS)):

        if proximo != no and CUSTOS[no][proximo] != np.inf:
            vizinhos.append(proximo)

    return vizinhos

## 3. Desafio 1 — Calcular a atratividade

Fórmula usada (a mesma do Laboratório 01):

atratividade = feromônio^ALPHA × (1/custo)^BETA


In [4]:
def calcular_atratividade(no_atual, proximo):

    fer = feromonio[no_atual][proximo]
    custo = CUSTOS[no_atual][proximo]

    atratividade = (fer ** ALPHA) * (1 / custo) ** BETA

    return atratividade

## 4. Desafio 2 — Evaporação

In [5]:
def evaporar_feromonio():

    global feromonio

    feromonio *= (1 - TAXA_EVAPORACAO)

    feromonio[CUSTOS == np.inf] = 0

## 5. Desafio 3 — Depósito

Quanto menor o custo da rota, maior o depósito (`Q / custo`).

In [6]:
def depositar_feromonio(rota, custo):

    deposito = Q / custo

    for i in range(len(rota) - 1):

        origem = rota[i]
        destino = rota[i + 1]

        feromonio[origem][destino] += deposito

## 6. Desafio 4 — Construir uma rota

Reaproveitamos `calcular_atratividade` para montar a lista de atratividades, transformamos em probabilidades e sorteamos o próximo nó com `random.choices`, do mesmo jeito que no Laboratório 01.

In [7]:
def construir_rota():

    rota = [ORIGEM]
    atual = ORIGEM

    while atual != DESTINO:

        vizinhos = obter_vizinhos(atual)

        candidatos = [
            no for no in vizinhos
            if no not in rota
        ]

        if not candidatos:
            return None

        atratividades = [
            calcular_atratividade(atual, proximo)
            for proximo in candidatos
        ]

        soma = sum(atratividades)
        probabilidades = [valor / soma for valor in atratividades]

        proximo = random.choices(
            candidatos,
            weights=probabilidades,
            k=1
        )[0]

        rota.append(proximo)
        atual = proximo

    return rota

### Observação

O roteiro usa `calcular_custo(rota)` na etapa de execução, mas essa função não é redefinida no Laboratório 03 — ela é a mesma função do Laboratório 01, reaproveitada aqui.

In [8]:
def calcular_custo(rota):

    total = 0

    for i in range(len(rota) - 1):

        origem = rota[i]
        destino = rota[i + 1]

        total += CUSTOS[origem][destino]

    return total

## 7. Execução

In [9]:
melhor_rota = None
melhor_custo = float("inf")

for iteracao in range(NUM_ITERACOES):

    rotas = []

    for _ in range(NUM_FORMIGAS):

        rota = construir_rota()

        if rota is not None:

            custo = calcular_custo(rota)

            rotas.append((rota, custo))

            if custo < melhor_custo:

                melhor_custo = custo
                melhor_rota = rota.copy()

    evaporar_feromonio()

    for rota, custo in rotas:

        depositar_feromonio(
            rota,
            custo
        )

print("Melhor rota:", melhor_rota)
print("Melhor custo:", melhor_custo)

Melhor rota: [0, 1, 2, 3, 4, 5]
Melhor custo: 8.0
